# Efficient Drug Discovery using Molecular Data
## Notebook 2: Preprocessing & Feature Engineering
**Author:** Dev Kapania | IIT Roorkee Research Intern

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
import joblib
import os
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded!')

## 1. Load Raw Data

In [ ]:
df = pd.read_csv('../data/raw/drug_data.csv')
print(f'Raw data shape: {df.shape}')
df.head()

## 2. Handle Missing Values

In [ ]:
print(f'Missing values before: {df.isnull().sum().sum()}')

# Drop columns with >50% missing
threshold = len(df) * 0.5
df = df.dropna(thresh=threshold, axis=1)

# Fill remaining nulls with median
for col in df.select_dtypes(include=np.number).columns:
    df[col].fillna(df[col].median(), inplace=True)

print(f'Missing values after:  {df.isnull().sum().sum()}')
print(f'Shape after cleaning:  {df.shape}')

## 3. Remove Low-Variance Features

In [ ]:
from sklearn.feature_selection import VarianceThreshold

X = df.drop('activity', axis=1)
y = df['activity']

selector = VarianceThreshold(threshold=0.01)
X_filtered = selector.fit_transform(X)
selected_cols = X.columns[selector.get_support()]

print(f'Features before variance filter: {X.shape[1]}')
print(f'Features after  variance filter: {X_filtered.shape[1]}')

X = pd.DataFrame(X_filtered, columns=selected_cols)

## 4. Select Top K Features (ANOVA F-test)

In [ ]:
K = min(50, X.shape[1])  # top 50 features
selector_k = SelectKBest(f_classif, k=K)
X_selected = selector_k.fit_transform(X, y)
top_features = X.columns[selector_k.get_support()]

print(f'Selected top {K} features using ANOVA F-test')
X = pd.DataFrame(X_selected, columns=top_features)

# Save feature selector
os.makedirs('../models', exist_ok=True)
joblib.dump(selector_k, '../models/feature_selector.pkl')
print('Feature selector saved!')

## 5. Train/Test Split & Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

# Validation split from train
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

joblib.dump(scaler, '../models/scaler.pkl')

print(f'Train: {X_train_scaled.shape}')
print(f'Val:   {X_val_scaled.shape}')
print(f'Test:  {X_test_scaled.shape}')

os.makedirs('../data/processed', exist_ok=True)
np.save('../data/processed/X_train.npy', X_train_scaled)
np.save('../data/processed/X_val.npy',   X_val_scaled)
np.save('../data/processed/X_test.npy',  X_test_scaled)
np.save('../data/processed/y_train.npy', y_train.values)
np.save('../data/processed/y_val.npy',   y_val.values)
np.save('../data/processed/y_test.npy',  y_test.values)
print('All splits saved!')